In [ ]:
!pip install qiskit qiskit-aer qiskit-ionq matplotlib networkx numpy -q

# Calibration-Aware Hardware Ranking for Quantum Circuits

Running a quantum circuit is not like running a classical program. Different quantum backends have different calibration states every day — qubits vary in readout fidelity, gate error, and coherence time. Choosing the wrong backend can cost you fidelity before compilation even starts.

This notebook demonstrates **calibration-aware backend ranking**: a systematic way to select the best available backend for your circuit, using real calibration data rather than static hardware specs.

**What we cover, end to end:**
load calibration → normalize → extract features → score → rank → visualize → explain → run on real hardware

Three worked example circuits (Bell, GHZ(5), QFT(4)) and a bring-your-own-circuit section are included. No credentials needed to run ranking and visualization — only the final hardware run cell requires a Qollab account.

In [ ]:
import json, math
from pathlib import Path
from statistics import mean, median

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
from qiskit import QuantumCircuit

from utils import (
    IQMAdapter, IBMAdapter, IonQAdapter,
    per_qubit_health, backend_health_score,
    print_health_summary,
    bell_circuit, ghz_circuit, qft_circuit,
    extract_circuit_features,
    rank_backends, print_ranking_table, DEFAULT_WEIGHTS,
    plot_backend_health, plot_backends_side_by_side,
    explain_ranking,
)

---
## Part 1 — Loading Calibration Data

Every provider returns calibration in a different format. IQM reports per-qubit SSRO (state-space readout) fidelity. IBM reports T1, T2, and readout error. IonQ reports SPAM fidelity and 1Q/2Q gate fidelity.

We normalize all three into a common schema:

```
BackendSnapshot
  backend_name  : str
  provider      : str
  num_qubits    : int
  qubits        : list[QubitSnapshot]
  connectivity  : list[tuple[str, str]]

QubitSnapshot
  qubit_id          : str
  readout_fidelity  : float   # 0–1, higher is better
  readout_error     : float   # 0–1, lower is better
  t1_us             : float | None
  t2_us             : float | None
```

Adding a new provider means writing one adapter. The scoring and ranking code never changes.

In [ ]:
# IQM Sirius — real calibration snapshot from the IQM API
# SSRO fidelity per qubit; no T1/T2 in this snapshot format
iqm = IQMAdapter.load("data/iqm_sirius.json")

print(f"Backend : {iqm.backend_name} ({iqm.provider})")
print(f"Qubits  : {iqm.num_qubits}")
print(f"Fetched : {iqm.timestamp}")
print(f"\nSample qubit: {iqm.qubits[0]}")

In [ ]:
# IBM Kolkata — synthetic calibration matching published IBM Eagle-class specs
# T1, T2, and readout error per qubit
ibm = IBMAdapter.load("data/ibm_kolkata_fake.json")

print(f"Backend : {ibm.backend_name} ({ibm.provider})")
print(f"Qubits  : {ibm.num_qubits}")
print(f"\nSample qubit: {ibm.qubits[0]}")

In [ ]:
# IonQ Aria 1 and Aria 2 — real calibration from the IonQ API
# SPAM fidelity + 1Q/2Q gate fidelity; T1/T2 in seconds (trapped-ion)
# All-to-all connectivity: every qubit pair can interact natively
aria1 = IonQAdapter.load("data/ionq_aria_1.json")
aria2 = IonQAdapter.load("data/ionq_aria_2.json")

for backend in [aria1, aria2]:
    fid = backend.metadata.get("fidelity", {})
    print(f"Backend : {backend.backend_name} ({backend.provider})")
    print(f"Qubits  : {backend.num_qubits}  |  Topology: {backend.metadata['topology']}")
    print(f"SPAM    : {fid.get('spam')}  |  1Q: {fid.get('1q')}  |  2Q: {fid.get('2q')}")
    print(f"T1      : {backend.qubits[0].t1_us / 1e6:.0f}s  |  T2: {backend.qubits[0].t2_us / 1e6:.1f}s")
    print()

---
## Part 2 — Per-Qubit Health

Raw calibration metrics mean different things on different backends. We normalize everything into a single health score per qubit in [0, 1]:

- **IQM**: `health = SSRO fidelity` (the dominant readout metric)
- **IBM**: `health = 0.70 × readout_fidelity + 0.20 × T1_normalized + 0.10 × T2_normalized`
- **IonQ**: `health = 0.50 × SPAM + 0.30 × 1Q_gate_fidelity + 0.20 × 2Q_gate_fidelity`

The health score tells you, at a glance, which qubits to prefer when mapping your circuit.

In [ ]:
# IQM — per-qubit health bar chart
iqm_health = per_qubit_health(iqm)

print(f"IQM Sirius — per-qubit health")
for qid, h in sorted(iqm_health.items(), key=lambda x: -x[1]):
    bar = "█" * int(h * 50)
    print(f"  {qid:6s}  {h:.4f}  {bar}")

In [ ]:
# IBM — per-qubit health bar chart (note: T1/T2 variation visible here)
ibm_health = per_qubit_health(ibm)

print(f"IBM Kolkata — per-qubit health (top 10)")
for qid, h in sorted(ibm_health.items(), key=lambda x: -x[1])[:10]:
    bar = "█" * int(h * 50)
    print(f"  Q{qid:3s}  {h:.4f}  {bar}")

In [ ]:
# Summary for all four backends
for backend in [iqm, ibm, aria1, aria2]:
    print_health_summary(backend)
    print()

---
## Part 3 — Example Circuits

We use three canonical circuits that test different hardware properties:

- **Bell** — 2-qubit entanglement. Minimal circuit; tests basic readout and 2Q gate quality.
- **GHZ(5)** — 5-qubit entanglement via a CNOT chain. Tests multi-qubit connectivity.
- **QFT(4)** — Quantum Fourier Transform. Tests long-range 2Q interactions and depth sensitivity.

The ranking function extracts three features from each circuit: qubit count (capacity check), depth (coherence pressure), and 2Q gate count (entanglement demand).

In [ ]:
bell = bell_circuit()
ghz  = ghz_circuit(5)
qft  = qft_circuit(4)

for qc in [bell, ghz, qft]:
    print(f"\n── {qc.name} ──")
    print(qc.draw("text", fold=80))

In [ ]:
print(f"{'Circuit':<12} {'Qubits':>7} {'Depth':>7} {'2Q gates':>10} {'Ent. ratio':>12}")
print("-" * 52)
for qc in [bell, ghz, qft]:
    f = extract_circuit_features(qc)
    print(f"{f['name']:<12} {f['num_qubits']:>7} {f['depth']:>7} "
          f"{f['two_qubit_count']:>10} {f['entanglement_ratio']:>12.2f}")

---
## Part 4 — Scoring and Ranking

The scoring function combines three components:

```
score = 0.70 × readout_quality
      + 0.20 × coherence_margin
      + 0.10 × capacity_fit
```

**readout_quality** — mean health of the N best qubits (N = circuit width). Primary differentiator for shallow NISQ circuits.

**coherence_margin** — T1 relative to estimated circuit runtime. Matters for deeper circuits where decoherence becomes a real risk.

**capacity_fit** — 1.0 if the backend has enough physical qubits; 0.0 if not. Hard fail.

Weights are named constants (`DEFAULT_WEIGHTS`) — easy to adjust. No learned parameters.

In [ ]:
# Bell circuit — all four backends
all_backends = [iqm, ibm, aria1, aria2]
results_bell = rank_backends(all_backends, bell)
print_ranking_table(results_bell)

In [ ]:
# GHZ(5) — five-qubit entanglement
results_ghz = rank_backends(all_backends, ghz)
print_ranking_table(results_ghz)

In [ ]:
# QFT(4) — depth-sensitive circuit
results_qft = rank_backends(all_backends, qft)
print_ranking_table(results_qft)

### What happens when the circuit is too large?

IQM Sirius has 16 qubits. If a circuit needs more, `capacity_fit = 0.0` — a hard fail regardless of how good the calibration is.

In [ ]:
# 20-qubit circuit — exceeds IQM Sirius capacity
large_qc = QuantumCircuit(20)
large_qc.h(range(20))
large_qc.measure_all()

results_large = rank_backends(all_backends, large_qc)
print_ranking_table(results_large)

---
## Part 5 — Topology Visualization

Each node is a physical qubit, colored by health score (green = healthy, red = degraded). Edges show which qubits can interact natively.

IQM and IBM have fixed connectivity graphs (star and heavy-hex topology). IonQ Aria is all-to-all — every qubit pair can interact via the native MS gate, so it's shown as a ring with no routing overhead.

In [ ]:
# IQM and IBM — fixed topology graphs
plot_backends_side_by_side([iqm, ibm])

In [ ]:
# IonQ Aria 1 and Aria 2 — all-to-all, shown as circular layout
# Note the health score difference: Aria 1 was in a degraded state at calibration time
plot_backends_side_by_side([aria1, aria2])

---
## Part 6 — Reasoning Trace

A score alone doesn't tell you *why* one backend beat another. `explain_ranking()` generates a deterministic plain-language explanation from the scoring breakdown — no language model, fully reproducible.

In [ ]:
# Why did Aria 2 rank first for the Bell circuit?
print(explain_ranking(results_bell))

In [ ]:
# Why does IQM lose on a 20-qubit circuit?
print(explain_ranking(results_large))

---
## Part 7 — Bring Your Own Circuit

Replace the circuit below with your own. The ranking and reasoning trace update automatically.

In [ ]:
# ── Edit this circuit ────────────────────────────────────────────────────
my_qc = QuantumCircuit(3)
my_qc.h(0)
my_qc.cx(0, 1)
my_qc.cx(1, 2)
my_qc.rz(0.5, 2)
my_qc.measure_all()

# ── Rank and explain ─────────────────────────────────────────────────────
my_results = rank_backends(all_backends, my_qc)
print_ranking_table(my_results)
print()
print(explain_ranking(my_results))

---
## Part 8 — Hardware Run on IonQ Aria 2

Aria 2 ranked #1 across all our example circuits. We'll run the Bell circuit on it and compare the hardware result against a noiseless simulation.

In [ ]:
# Noiseless simulation — always runs, no credentials needed
from qiskit_aer import AerSimulator

qc_bell = bell_circuit()
sim = AerSimulator()
job_sim = sim.run(qc_bell, shots=1024)
counts_sim = job_sim.result().get_counts()
print("Noiseless simulation:")
print(counts_sim)

In [ ]:
# Hardware run on IonQ Aria 2 via Qollab playground runner
# Credentials are managed by the Qollab environment
try:
    from qiskit_ionq import IonQProvider
    from qiskit import transpile

    provider = IonQProvider()  # picks up credentials from Qollab environment
    backend_hw = provider.get_backend("ionq_qpu.aria-2")

    qc_transpiled = transpile(qc_bell, backend_hw)
    job_hw = backend_hw.run(qc_transpiled, shots=1024)
    print(f"Job submitted — ID: {job_hw.job_id()}")
    print("Waiting for results (this may take a few minutes)...")

    counts_hw = job_hw.result().get_counts()
    print("\nHardware result:")
    print(counts_hw)

except Exception as e:
    print(f"Hardware run error: {e}")
    print("If running locally, set IONQ_API_KEY in your environment.")
    counts_hw = None

In [ ]:
# Compare simulation vs hardware
if counts_hw:
    labels = sorted(set(counts_sim) | set(counts_hw))
    x = range(len(labels))

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar([i - 0.2 for i in x], [counts_sim.get(l, 0) for l in labels],
           width=0.4, label="Simulation", color="steelblue", alpha=0.8)
    ax.bar([i + 0.2 for i in x], [counts_hw.get(l, 0) for l in labels],
           width=0.4, label="IonQ Aria 2", color="coral", alpha=0.8)
    ax.set_xticks(list(x))
    ax.set_xticklabels(labels)
    ax.set_xlabel("Bitstring")
    ax.set_ylabel("Counts (1024 shots)")
    ax.set_title("Bell circuit: Simulation vs IonQ Aria 2")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("Hardware counts not available — run the hardware cell first.")

---
## About QAIP

This notebook demonstrates calibration-aware backend ranking — one primitive in a larger project called [QAIP](https://github.com/Rudra1x/QAIP), which builds on top of this to also handle qubit mapping, routing cost estimation, and full execution planning.

If you found this useful, check it out: **[github.com/Rudra1x/QAIP](https://github.com/Rudra1x/QAIP)**